# Music Metadata Ingestion Pipeline

Ingests structured music metadata from a semicolon-delimited CSV stored in a Unity Catalog volume, enriches it with bronze-layer audit columns, and persists the result to a managed Delta table.

**Pipeline steps:**
* Restart Python and load the shared `music_pipeline_setup` module from `00_setup/pawel_project`
* Ensure volume subdirectories (`jsons/`, `music_metadata/`) exist
* Define the expected CSV schema (avoids costly inference, enforces date types)
* Read `music_discography.csv` from the Unity Catalog volume
* Enrich with `ingest_timestamp` and `ingest_file` bronze audit columns
* Preview the result, then write to the silver Delta table

**Source:** `/Volumes/dbr_dev/music_analytics/raw_landing_zone/music_metadata/music_discography.csv`  
**Target table:** `config.silver_music_metadata_table`

**Note**: The table can be already viewed as a silver table - it has no duplicates, no missing values, dtypes are correct.

## Step 0 – importing all libraries 
The module (in `00_setup/pawel_project/music_pipeline_setup.py`) centralises all path constants, catalog/schema names, and setup helper functions used across the pipeline.

In [0]:
# Restart the Python interpreter to clear any stale library versions from the session.
# This must run before any imports to ensure the cleanest possible environment.
dbutils.library.restartPython()

In [0]:
import sys
# music_pipeline_setup.py lives two levels up in 00_setup/pawel_project.
# Relative imports are unsupported in Databricks notebooks, and '00_setup' starts
# with digits (invalid Python identifier), so we add the directory to sys.path
# and import config as a regular module.
sys.path.insert(0, "../../00_setup/pawel_project")

from music_pipeline_setup import (
    music_metadata_file,
    silver_music_metadata_table,
)
from pyspark.sql.types import (
    StructType, 
    StructField, 
    StringType, 
    DateType
)
from pyspark.sql.functions import current_timestamp, col, to_date

## Step 1 — Environment Setup & Configuration

In [0]:
# Explicit schema for the music metadata CSV (we declare every column to be of SringType - in the further cells we'll cast the columns to more relevant dtypes)
schema = StructType([
    StructField("url", StringType(), True),               # YouTube video URL
    StructField("title", StringType(), True),             # Song / video title
    StructField("album", StringType(), True),             # Album name
    StructField("album_release_date", StringType(), True),   # Individual song release date
    StructField("author", StringType(), True)  # Full album release date
])

## Step 2 — Read CSV, Enrich & Write to Bronze Table

Read the source CSV from the Unity Catalog volume using the predefined schema, enrich with standard bronze audit columns (`ingest_timestamp`, `ingest_file`), preview the output, then persist to the managed Delta table.

In [0]:
# Read the music metadata CSV from the Unity Catalog volume.
# Key options:
#   header=True       — first row contains column names
#   sep=";"           — semicolon delimiter (not the default comma)
#   encoding="UTF-8"  — handles special characters in titles/album names
#   schema=schema     — uses the pre-defined schema; avoids inference and type mismatches
#   dateFormat        — dates are in European DD.MM.YYYY format
music_metadata_df = spark.read.csv(
    music_metadata_file,
    header=True,
    sep=";",
    encoding="UTF-8",
    schema=schema,
    dateFormat="dd.MM.yyyy"
)

In [0]:
# Enrich the raw DataFrame with standard bronze-layer audit columns:
#   ingest_timestamp — wall-clock time of this pipeline run
#   ingest_file      — full path of the source file, read from Spark's built-in
#                      _metadata virtual column (available for all file-based sources)
columns_to_add = {
    "ingest_timestamp": current_timestamp(),
    "ingest_file": col("_metadata.file_path")
}

music_metadata_df_bronze = (music_metadata_df
                            .withColumns(columns_to_add)
                            .withColumn("album_release_date", to_date(col("album_release_date"), "dd.MM.yyyy")))

In [0]:
# Sanity check: display 2 rows to verify schema, data values,
# and that the bronze audit columns (ingest_timestamp, ingest_file) were added correctly.
music_metadata_df_bronze.show(2)

In [0]:
# Write the enriched bronze DataFrame to the managed Delta table.
#   mode("overwrite")        — full refresh; replaces all existing rows on each run
#   overwriteSchema("true")  — allows schema evolution if columns are added or renamed
# The table name is centralised in config to keep it consistent across notebooks.
(music_metadata_df_bronze
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(silver_music_metadata_table)
)